# Начало работы с TaqSim: река, водохранилище и хозяйство

Этот блокнот работает в **Google Colab** с бесплатной учётной записью Google. На ноутбук ничего устанавливать не нужно.

Вы выполните три шага:

1. **Установите TaqSim** (около минуты) и создадите небольшую водную систему типа Зарафшана: реку, гидроузел, отводящий воду в канал, водохранилище на канале, орошаемый район и продолжение реки ниже по течению.
2. **Выполните расчёт** за три года и посмотрите, как меняется запас водохранилища и получает ли хозяйство нужную воду.
3. **Передайте задачу языковой модели**: попросите Gemini (встроенную в Colab) или другую бесплатную чат-модель изменить модель.

Запускайте ячейки по порядку (можно выбрать `Runtime → Run all`, «Среда выполнения → Выполнить все»). Если ячейка выдаёт ошибку, прочитайте сообщение и перейдите к разделу «Передайте задачу языковой модели», где объясняется, как её исправить.

<p><img src="https://hydrosolutions.github.io/zarafshan-taqsim-course/images/hydrosolutions_logo.png" alt="hydrosolutions" width="220"></p>
<p><b>hydrosolutions GmbH</b> · Venusstrasse 29, 8050 Zürich, Switzerland · <a href="https://www.hydrosolutions.ch">www.hydrosolutions.ch</a> · hs@hydrosolutions.ch · +41 43 535 05 80</p>

In [ ]:
# @title Шаг 1: установка TaqSim v0.1.4 (чистый Python, компилятор не нужен)....
# Шаг 1: установка TaqSim v0.1.4 (чистый Python, компилятор не нужен). Около минуты в Colab.
# В Colab уже есть numpy, pandas, matplotlib и другие пакеты. Добавляем два недостающих пакета, затем TaqSim
# без списка зависимостей, чтобы не менять пакеты Jupyter в Colab.
%pip install -q deap ctrl-freak
%pip install -q --no-deps "git+https://github.com/hydrosolutions/taqsim.git@v0.1.4"

import sys
import taqsim

print("TaqSim", taqsim.__version__, "на Python", sys.version.split()[0])

## Система

```
river  ──▶  headworks  ──▶  river_downstream        (неотведённая вода остаётся в реке)
               │
               └──▶  reservoir  ──▶  farm  ──▶  drain
```

- **river**: месячный приток в Раватходже, млн м³ (Mm³) в месяц. Значения — медианы каждого календарного месяца за 2010–2023 годы, рассчитанные по суточным наблюдениям гидропоста, используемым в курсе.
- **headworks**: распределитель. Фиксированная *доля* речной воды поступает в канал, питающий водохранилище; остальное течёт дальше.
- **reservoir**: вместимость 667 млн м³, из них 66.7 млн м³ — мёртвый объём (ниже самого низкого выпуска). В начале гидрологического года, 1 октября, запас составляет 170 млн м³. Эти значения используются для Каттакургана в учебной игре.
- **farm**: Каттакурганский район орошения с потребностью 385 млн м³ в год, почти полностью с апреля по сентябрь.
- **drain** и **river_downstream**: места выхода воды из модели.

Все величины задаются **за шаг времени**: один шаг — месяц, все объёмы — млн м³ в месяц. Правило водохранилища простое: *выпускать месячную потребность хозяйства, пока есть вода выше мёртвого объёма*.

Год идёт с октября по сентябрь (гидрологический год), поэтому индекс 0 означает октябрь.

In [ ]:
# @title Система · код
MONTHS = ["Oct", "Nov", "Dec", "Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep"]

# Приток Раватходжи, млн м³ в месяц, медиана 2010–2023, начиная с октября.
INFLOW_MM3 = (203.0, 162.9, 133.5, 124.3, 106.2, 124.2, 155.6, 381.5, 771.1, 993.4, 711.9, 350.5)

# Потребность Каттакурганского района, млн м³ в месяц, начиная с октября.
FARM_NEED_MM3 = (18.8, 0.0, 0.0, 0.0, 0.0, 3.3, 18.5, 38.6, 43.0, 83.0, 113.4, 66.8)

YEARS = 3
CANAL_SHARE = 0.08  # доля речного стока, отводимая в канал у гидроузла

print(f"Река приносит {sum(INFLOW_MM3):,.0f} млн м³ в год; хозяйству нужно {sum(FARM_NEED_MM3):,.0f} млн м³ в год.")

## Два правила

Узлы TaqSim сами не принимают решений. Вы задаёте им *правило* (небольшой класс Python):

- гидроузлу нужна **политика распределения**: сколько воды направлять в каждую ветвь;
- водохранилищу нужна **политика выпуска**: сколько выпускать каждый месяц.

Правило — это неизменяемый класс данных (`frozen dataclass`) с одним методом. Поля, перечисленные в `__params__`, — параметры, которые позже сможет менять оптимизатор.

In [ ]:
# @title Два правила · код
from dataclasses import dataclass
from typing import ClassVar

from taqsim import Splitter, Storage, Strategy, Timestep


@dataclass(frozen=True)
class FixedShare(Strategy):
    '''Направить фиксированную долю в канал; остальное остаётся в реке.'''

    __params__: ClassVar[tuple[str, ...]] = ("share",)
    __bounds__: ClassVar[dict[str, tuple[float, float]]] = {"share": (0.0, 1.0)}
    share: float = 0.1

    def split(self, node: Splitter, amount: float, t: Timestep) -> dict[str, float]:
        return {"reservoir": amount * self.share, "river_downstream": amount * (1.0 - self.share)}


@dataclass(frozen=True)
class ReleaseTheNeed(Strategy):
    '''Выпустить месячную потребность хозяйства, не превышая доступный запас выше мёртвого объёма.'''

    __params__: ClassVar[tuple[str, ...]] = ()

    def release(self, node: Storage, inflow: float, t: Timestep) -> float:
        need = FARM_NEED_MM3[t.index % 12]
        return min(need, node.storage - node.dead_storage)

## Соберите систему

Узлы обрабатывают воду. Рёбра определяют, кто кому её передаёт. `validate()` проверяет полноту сети.

In [ ]:
# @title Сборка системы · код
from taqsim import Demand, Edge, Frequency, Sink, Source, TimeSeries, WaterSystem
from taqsim.node import NoLoss


def build_system(canal_share: float = CANAL_SHARE, years: int = YEARS) -> WaterSystem:
    # location=(lat, lon) задаёт реальные координаты; здесь служит лишь для размещения узлов на схеме.
    system = WaterSystem(frequency=Frequency.MONTHLY)
    system.add_node(Source(id="river", inflow=TimeSeries(values=list(INFLOW_MM3) * years), location=(1.0, 0.0)))
    system.add_node(Splitter(id="headworks", split_policy=FixedShare(share=canal_share), location=(1.0, 1.0)))
    system.add_node(
        Storage(
            id="reservoir",
            capacity=667.0,
            dead_storage=66.7,
            initial_storage=170.0,
            release_policy=ReleaseTheNeed(),
            loss_rule=NoLoss(),  # испарения пока нет: см. упражнение 2 ниже
            location=(0.4, 2.0),
        )
    )
    system.add_node(Demand(id="farm", requirement=TimeSeries(values=list(FARM_NEED_MM3) * years), location=(0.4, 3.0)))
    system.add_node(Sink(id="drain", location=(0.4, 4.0)))
    system.add_node(Sink(id="river_downstream", location=(1.6, 3.0)))
    for source, target in (
        ("river", "headworks"),
        ("headworks", "reservoir"),
        ("headworks", "river_downstream"),
        ("reservoir", "farm"),
        ("farm", "drain"),
    ):
        system.add_edge(Edge(id=f"{source}_to_{target}", source=source, target=target))
    system.validate()
    return system


system = build_system()
fig, ax = system.visualize(figsize=(8, 3.5), title="Начальная система")

## Выполните расчёт и прочитайте результат

При расчёте каждый узел записывает *события*: получение, накопление, выпуск и перелив воды, а также дефицит в хозяйстве. Чтобы получить результат, нужно суммировать нужные события отдельно за каждый месяц.

In [ ]:
# @title Расчёт и чтение результата · код
import pandas as pd
from taqsim.node import DeficitRecorded, WaterLost, WaterReleased, WaterSpilled, WaterStored


def monthly(node, event_type, field: str = "amount", steps: int = 12 * YEARS) -> list[float]:
    totals = [0.0] * steps
    for event in node.events_of_type(event_type):
        totals[event.t] += getattr(event, field)
    return totals


def run(canal_share: float = CANAL_SHARE) -> pd.DataFrame:
    system = build_system(canal_share)
    system.simulate(12 * YEARS)
    reservoir, farm = system.nodes["reservoir"], system.nodes["farm"]
    stored = monthly(reservoir, WaterStored)
    lost = monthly(reservoir, WaterLost)
    released = monthly(reservoir, WaterReleased)
    storage, level = [], reservoir.initial_storage
    for s, l, r in zip(stored, lost, released):
        level += s - l - r
        storage.append(level)
    return pd.DataFrame(
        {
            "year": [i // 12 + 1 for i in range(12 * YEARS)],
            "month": MONTHS * YEARS,
            "river_in": list(INFLOW_MM3) * YEARS,
            "to_canal": [v * canal_share for v in INFLOW_MM3] * YEARS,
            "spilled": monthly(reservoir, WaterSpilled),
            "storage_end": storage,
            "farm_need": list(FARM_NEED_MM3) * YEARS,
            "shortage": monthly(farm, DeficitRecorded, "deficit"),
        }
    ).round(1)


result = run()
print(f"Суммарный дефицит за {YEARS} года: {result.shortage.sum():.1f} млн м³; "
      f"конечный запас {result.storage_end.iloc[-1]:.1f} млн м³.")
result[result.year == YEARS]

In [ ]:
# @title Расчёт и чтение результата · код
import matplotlib.pyplot as plt

fig, (top, bottom) = plt.subplots(2, 1, figsize=(9, 5.5), sharex=True)
top.plot(result.storage_end, color="tab:blue")
top.axhline(66.7, color="grey", ls="--", lw=1)
top.set_ylabel("запас, млн м³")
top.set_title(f"Доля канала {CANAL_SHARE:.0%}: запас уменьшается за три года")
bottom.bar(result.index, result.farm_need, color="lightgrey", label="потребность хозяйства")
bottom.bar(result.index, result.shortage, color="tab:red", label="дефицит")
bottom.set_ylabel("млн м³ в месяц")
bottom.set_xticks(range(0, 12 * YEARS, 3))
bottom.set_xticklabels([f"{MONTHS[i % 12]} y{i // 12 + 1}" for i in range(0, 12 * YEARS, 3)])
bottom.legend(loc="upper left")
plt.tight_layout()

## Передайте задачу языковой модели

Не следует рассчитывать, что языковая модель уже знает TaqSim. Она может написать правильный код TaqSim, **если в блокноте есть работающие примеры и приведённая ниже памятка**. Поэтому блокнот такой подробный: он служит справочником для модели.

**В Colab**: нажмите синюю кнопку **Gemini** внизу по центру страницы (или значок искры в ячейке кода) и опишите задачу обычными словами. Gemini видит предыдущие ячейки. Вставьте предложенный код в новую ячейку, запустите и, если появится ошибка, отправьте её текст обратно.

Попробуйте по одному запросу:

1. *«Найди минимальную долю отвода в канал с шагом 1 процент, при которой в хозяйстве нет дефицита ни в одном из трёх лет. Используй функцию run().»*
2. *«Добавь испарение из водохранилища: каждый месяц теряется доля его запаса. Напиши класс потерь по образцу NoLoss с месячными долями, начиная с октября: 1.6, 1.0, 0.6, 0.6, 0.6, 1.2, 2.1, 4.4, 6.6, 7.6, 6.2, 4.1 процента. Повтори расчёт и покажи, насколько вырос дефицит.»*
3. *«Добавь город между водохранилищем и хозяйством. Ему нужно 6 млн м³ каждый месяц; он обслуживается раньше хозяйства и возвращает 70 процентов воды в реку. Используй узел Demand с consumption_fraction.»*
4. *«Замени типичный год маловодным: умножь приток на 0.85. Какая доля отвода в канал теперь нужна хозяйству?»*

Затем задайте свой вопрос. Хороший запрос описывает **поведение воды**, а не требуемый программный код.

**Вне Colab** (приложение Gemini, ChatGPT Free, claude.ai Free): вставьте в чат памятку ниже и ячейку, которую хотите изменить, попросите новую ячейку, перенесите её сюда и запустите. Цикл тот же: передать контекст, получить код, вернуть сообщение об ошибке.

## Памятка по TaqSim (v0.1.4)

**Импорт**
```python
from taqsim import (Demand, Edge, Frequency, PassThrough, Sink, Source, Splitter, Storage,
                    Strategy, TimeSeries, Timestep, WaterSystem)
from taqsim.node import (NoLoss, DeficitRecorded, WaterReceived, WaterReleased, WaterSpilled,
                         WaterStored, WaterLost, WaterConsumed, WaterDistributed, EVAPORATION, SEEPAGE)
```

**Узлы** (каждому нужен уникальный `id`; можно задать `location=(lat, lon)`; получателей определяют рёбра, а не узлы):

| Узел | Обязательные аргументы | Что делает |
|---|---|---|
| `Source(id, inflow=TimeSeries)` | приток за шаг | вводит воду в систему |
| `Splitter(id, split_policy=...)` | правило распределения | направляет доли нескольким получателям |
| `Storage(id, capacity, release_policy=..., loss_rule=..., initial_storage=0, dead_storage=0)` | вместимость и оба правила | водохранилище; избыток сверх вместимости переливается |
| `Demand(id, requirement=TimeSeries, consumption_fraction=1.0, efficiency=1.0)` | потребность за шаг | забирает нужное, передаёт остаток дальше |
| `PassThrough(id, capacity=None)` | нет | соединение или турбина; избыток сверх пропускной способности сбрасывается |
| `Sink(id)` | нет | конец системы |

**Рёбра**: `system.add_edge(Edge(id="a_to_b", source="a", target="b"))`. У каждого узла, кроме Sink, должно быть ровно одно исходящее ребро; исключение — Splitter, у которого одно ребро на каждого получателя. От каждого узла должен существовать путь к Sink.

**Система**: `system = WaterSystem(frequency=Frequency.MONTHLY)` (либо `Frequency.DAILY`), затем `add_node`, `add_edge`, `validate()`, `simulate(n_steps)`. Все значения задаются за шаг; TaqSim не переводит единицы. `TimeSeries(values=[...])` требует по одному неотрицательному значению на шаг. Для нового расчёта создайте новую систему (также работает `system.reset()`).

**Правила** — неизменяемые классы данных с одним методом; `t.index` — номер шага (0, 1, 2, ...):
```python
@dataclass(frozen=True)
class MyRelease(Strategy):                      # Storage.release_policy
    __params__: ClassVar[tuple[str, ...]] = ("rate",)
    rate: float = 10.0
    def release(self, node: Storage, inflow: float, t: Timestep) -> float:
        return min(self.rate, node.storage - node.dead_storage)

@dataclass(frozen=True)
class MySplit(Strategy):                        # Splitter.split_policy; ключи — id узлов-получателей
    __params__: ClassVar[tuple[str, ...]] = ("share",)
    share: float = 0.5
    def split(self, node: Splitter, amount: float, t: Timestep) -> dict[str, float]:
        return {"left": amount * self.share, "right": amount * (1 - self.share)}

@dataclass(frozen=True)
class MyLoss:                                   # Storage.loss_rule: НЕ Strategy; возвращает {причина: объём}
    share: float = 0.02
    def calculate(self, node: Storage, t: Timestep) -> dict:
        return {EVAPORATION: node.storage * self.share}
```

**Чтение результатов**: каждый узел хранит события. `node.events_of_type(WaterStored)` возвращает список; у каждого события есть `.amount` и `.t` (у `DeficitRecorded` — `.required`, `.actual`, `.deficit`). Суммируйте по `t`, чтобы получить месячный ряд. `system.nodes["reservoir"].storage` — запас после расчёта. `system.visualize()` рисует сеть.

**Порядок операций Storage на шаге**: получение, накопление (с переливом сверх вместимости), потери, выпуск. Demand забирает `min(received, requirement)`, потребляет долю `consumption_fraction`, передаёт остаток и избыток вниз. Воду ниже `dead_storage` нельзя выпустить, но она может испаряться.

**Частые ошибки**: забыто ребро к Sink (validate выдаёт ошибку); правило распределения называет получателя без ребра; отрицательные значения или NaN в TimeSeries; смешение единиц (м³/с и млн м³ в месяц); обращение к ряду цели через `trace[t]` вместо `.get(t, 0.0)`.

## Полная документация для более долгой работы в чате

Документация TaqSim входит в пакет. Следующая ячейка собирает её в ZIP-архив, который можно скачать и передать в чат, принимающий файлы (Gemini, ChatGPT и Claude поддерживают это). В исходном описании также упоминается вывод первого руководства для Gemini; эта ячейка фактически выводит список файлов архива.

In [ ]:
# @title Полная документация для чата · код
import shutil
from pathlib import Path
from taqsim import get_docs_path

docs = Path(get_docs_path())
archive = shutil.make_archive("taqsim_docs", "zip", docs)
files = sorted(p.relative_to(docs) for p in docs.rglob("*.md"))
print(f"В архиве {archive} файлов документации: {len(files)}:")
print("\n".join(f"  {p}" for p in files))

try:
    from google.colab import files as colab_files  # доступно только в Colab

    colab_files.download(archive)
except ImportError:
    pass

## Если Python установлен на вашем ноутбуке

Та же команда установки работает в среде с Python 3.12 или новее и `git`:

```
pip install "git+https://github.com/hydrosolutions/taqsim.git@v0.1.4"
```

После этого над блокнотом может работать агент программирования с открытым исходным кодом и доступной бесплатной моделью, например OpenCode с одной из бесплатных размещённых моделей. Возможности входа и бесплатного доступа у других средств, включая Codex CLI, зависят от действующих условий сервиса. Сохраните памятку выше в файле `AGENTS.md` в рабочей папке: так инструменты получают контекст о незнакомой библиотеке.